### Testing RAG Applications 📑

#### RAG Application
This application reads data about Model Context Protocol (MCP) server from internet, stores in vector stores, chunks the data with embedding and useful to answer the question about MCP while inferenced.

<img src="./img/RAG.png" width="500" height="400" style="display: block; margin: auto;">

In [27]:
!pip install  langchain_community langchain-chroma langchain-text-splitters langchain


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [28]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import List
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document
from langchain_ollama import ChatOllama

In [29]:
from dotenv import load_dotenv
import os

load = load_dotenv('./../.env')
os.environ.get("LOCAL_MODEL_BASE_URL")

'http://localhost:11434/v1'

In [20]:
llm = ChatOllama(
    base_url="http://localhost:11434",
    model = "gpt-oss:20b",
    temperature=0.5,
    max_tokens = 250
)

In [21]:
# Load data from Web
loader = WebBaseLoader("https://www.descope.com/learn/post/mcp")
data = loader.load()

# Split text into documents
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splits = text_splitter.split_documents(data)

# Add text to vector db
embedding = OllamaEmbeddings(model="qwen3-embedding:4b")
vectordb = Chroma.from_documents(documents=splits, embedding=embedding)

# Create a retriever
retriever = vectordb.as_retriever()


def format_docs(docs: List[Document]) -> str:
    return "\n\n".join([d.page_content for d in docs])


template = """Answer the question based only on the following context:

    {context}
    
    Give a summary not the full detail

    Question: {question}
    """
prompt = ChatPromptTemplate.from_template(template)


def retrieve_and_format(question):
    docs = retriever.invoke(question)
    return format_docs(docs)

chain = {"context": retrieve_and_format, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser()

#### Output of the LLM Application

In [22]:
response = chain.invoke("What is MCP")

print(response)

**MCP** is a communication framework that host AI applications (like Claude Desktop) use to talk to external tools or systems. It relies on the JSON‑RPC 2.0 standard to structure its messages—providing a clear, uniform format for requests, responses, and notifications—so the AI can quickly and seamlessly invoke outside tools from within the chat.


### Testing RAG Application with DeepEval
<img src="./img/RAGTesting.png" width="800" height="400" style="display: block; margin: auto;">

In [23]:
from deepeval.test_case import LLMTestCase
from deepeval.dataset import EvaluationDataset

test_case = LLMTestCase(
    input="What is MCP?",
    actual_output=chain.invoke("What is MCP"),
    expected_output="The Model Context Protocol (MCP) addresses this challenge by providing a standardized way for LLMs to connect with external data sources and tools—essentially a “universal remote” for AI apps"
)

dataset = EvaluationDataset()
dataset.add_test_case(test_case=test_case)

In [8]:
dataset

EvaluationDataset(test_cases=[LLMTestCase(input='What is MCP?', actual_output="MCP, or Model Context Protocol, is a protocol designed to facilitate interactions between large language models (LLMs) and various applications. It uses JSON-RPC 2.0 for standardized communication and supports two primary transport methods: STDIO for local integrations and HTTP+SSE for remote connections. MCP's architecture includes host applications that interact with users, MCP clients that handle communications with servers, and MCP servers that provide specific functions and context to AI apps. Overall, MCP enables seamless integration of LLMs into diverse applications by standardizing communication and providing a structured way to exchange data and context.", expected_output='The Model Context Protocol (MCP) addresses this challenge by providing a standardized way for LLMs to connect with external data sources and tools—essentially a “universal remote” for AI apps', context=None, retrieval_context=None

In [24]:
from deepeval.test_case import LLMTestCaseParams
from deepeval.metrics import GEval

concise_metrics = GEval(
    name = "Concise",
    criteria="Assess if the actual output remains concise while preserving all essential information.",
    
    evaluation_params=[
        LLMTestCaseParams.ACTUAL_OUTPUT
    ]
)

In [25]:
from deepeval.test_case import LLMTestCaseParams
from deepeval.metrics import GEval

completness_metrics = GEval(
    name = "Completeness",
    criteria="Assess whether the actual output retains all the key information from the input",
    
    evaluation_params=[
        LLMTestCaseParams.ACTUAL_OUTPUT
    ]
)

### Evaluation with GEval 

In [26]:
from deepeval.evaluate import evaluate
from deepeval.metrics import AnswerRelevancyMetric

evaluate(dataset.test_cases, metrics=[
    completness_metrics, 
    AnswerRelevancyMetric(),
    concise_metrics
])

✨ You're running DeepEval's latest Completeness [GEval] Metric! (using gpt-oss:20b (Local Model), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-oss:20b (Local Model), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Concise [GEval] Metric! (using gpt-oss:20b (Local Model), strict=False, 
async_mode=True)...

c:\Users\Navoki\PythonProjects\evluating_agent\.venv\Lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Metrics Summary

  - ✅ Completeness [GEval] (score: 1.0, threshold: 0.5, strict: False, evaluation model: gpt-oss:20b (Local Model), reason: The output reproduces the input text verbatim, ensuring all key facts, figures, and concepts are present and relationships preserved., error: None)
  - ✅ Answer Relevancy (score: 0.6666666666666666, threshold: 0.5, strict: False, evaluation model: gpt-oss:20b (Local Model), reason: The score is 0.67 because the answer includes an irrelevant 'JSON:' line and does not fully explain what MCP is, so it is partially relevant but not complete., error: None)
  - ❌ Concise [GEval] (score: 0.0, threshold: 0.5, strict: False, evaluation model: gpt-oss:20b (Local Model), reason: The evaluation requires comparing the word count and content to an original text, but the original text is not provided in the test case. Without that reference, it is impossible to assess length, essential points, or unnecessary additions, so the response cannot be evaluated again

⚠ WARNING: No hyperparameters logged.
» ]8;id=290585;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Done 🎉! View results on 
]8;id=749387;https://app.confident-ai.com/project/cmn70qogs003ilf1enonmdsks/test-runs/cmnakcqxz005drz1elp3ysmve/regression-testing\https://app.confident-ai.com/project/cmn70qogs003ilf1enonmdsks/test-runs/cmnakcqxz005drz1elp3ysmve/regression-testi]8;;\
]8;id=749387;https://app.confident-ai.com/project/cmn70qogs003ilf1enonmdsks/test-runs/cmnakcqxz005drz1elp3ysmve/regression-testing\ng]8;;\

EvaluationResult(test_results=[TestResult(name='test_case_0', success=False, metrics_data=[MetricData(name='Completeness [GEval]', threshold=0.5, success=True, score=1.0, reason='The output reproduces the input text verbatim, ensuring all key facts, figures, and concepts are present and relationships preserved.', strict_mode=False, evaluation_model='gpt-oss:20b (Local Model)', error=None, evaluation_cost=0.0, verbose_logs='Criteria:\nAssess whether the actual output retains all the key information from the input \n \nEvaluation Steps:\n[\n    "Extract all key facts, figures, and concepts from the input text.",\n    "Cross‑check each extracted key item against the actual output to confirm presence.",\n    "Verify that the context and relationships among key items are preserved in the output.",\n    "Assign a retention score or pass/fail based on the completeness of the key information."\n] \n \nRubric:\nNone \n \nScore: 1.0'), MetricData(name='Answer Relevancy', threshold=0.5, success=T